# Dock 04

## Configuration

In [1]:
#pull in local configuration
%run config.py
!cat config.py

# config.py
# This file wants to be listed in .gitignore

GNINA_LOC = "/home/dwaine/octoberproject/gnina"
GNINA_PARAMETER = "--no_gpu"


In [2]:
ligdictjson = "preprocessed/ligand.json"
prodictjson = "preprocessed/protein.json"

prodir = "preprocessed/proteinprep/"
dockeddir = "preprocessed/ligandprep/"
decoydir = "preprocessed/decoys/"
idealdir = "preprocessed/minimized_ideal_ligand/"
resultsdir = "results/"
decoy = "generated_decoys_activeER.sdf"
decoy = "generated_decoys_activeER_filtered_FINAL.sdf"
docked = resultsdir + "docked.sdf"
log = resultsdir + "gninalog.txt"
rmsdlog = resultsdir + "rmsd.txt"
results = resultsdir + "results.csv"

In [ ]:
sourceproteinprocesseddir = "../proteinprep01/chemfiles/"
sourceligandidealdir = "../mfaber_workflow/ER_Ligand_Prep/Ideal_Ligand/"
sourceligandidealdir = "../mfaber_workflow/ER_Ligand_Prep/minimized_ideal_ligand/"
localprocesseddir = "preprocessed/"

In [3]:
proteindict = {}
dockedliganddict = {}
idealliganddict = {}
gninaoptdict = {}
prepdict = {}
decoydf = None

In [4]:
import copy
def reportdict(rows, columns):
    lines = []
    if len(rows) == 0:
        return (lines) 
    inner_keys = set()
    for r in rows.values():
        inner_keys.update(r.keys())
    col_widths = {}
    col_widths["id"] = max(len("id"), max(len(k) for k in rows))
    for col in inner_keys:
        header_len = len(col)
        data_len = max(len(str(r.get(col, ""))) for r in rows.values())
        col_widths[col] = max(header_len, data_len)
    header = "  ".join(f"{col:<{col_widths[col]}}" for col in columns)
    lines.append(header)
    for outer_key, inner in rows.items():
        cells = [f"{outer_key:<{col_widths['id']}}"]
        for col in columns[1:]:
            cells.append(f"{str(inner.get(col, '')):<{col_widths[col]}}")
        lines.append("  ".join(cells))
    return (lines)

In [5]:
import time

def format_time(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"

In [ ]:
#copy preprocessed proteins and docked ligand files
!cp -R {sourceproteinprocesseddir}* {localprocesseddir}
!cp -R {sourceligandidealdir} {localprocesseddir}
!ls -al {localprocesseddir}

## Load and process decoy ligands from .sdf

In [ ]:
import pandas as pd
from rdkit.Chem import PandasTools

sdf_path = decoydir + decoy

decoydf = PandasTools.LoadSDF(
    sdf_path,
    molColName="ROMol",
    smilesName="SMILES",
    includeFingerprints=False,
    removeHs=False,
    strictParsing=True
)

# Add numbered ID column
decoydf["ID"] = [f"decoy{i+1}" for i in range(len(decoydf))]

print(decoydf.head())
print(decoydf.columns)
print(decoydf.shape)     # (rows, columns)
print(decoydf.info(memory_usage='deep'))

In [ ]:
from openbabel import openbabel as ob
from rdkit import Chem

def writeonedecoytoFS(row):
    
    # Your RDKit mol
    rdkit_mol = row['ROMol']

    # Convert RDKit MolBlock → Open Babel OBMol
    molblock = Chem.MolToMolBlock(rdkit_mol)
    ob_mol = ob.OBMol()
    ob_conversion = ob.OBConversion()
    ob_conversion.SetInFormat("mdl")  # MOL block format
    ob_conversion.ReadString(ob_mol, molblock)

    # Write out GNINA-ready SDF
    ob_conversion.SetOutFormat("sdf")
    sdf_string = ob_conversion.WriteString(ob_mol)

    with open(decoydir + "decoyligand.sdf", "w") as f:
        f.write(sdf_string)

## Populate docked ligand dictionary

In [6]:
dockedliganddict = {}

In [7]:
import json

# Read from text file
with open(ligdictjson, "r") as f:
    dockedliganddict = json.load(f)

#Which ligands are available?
lines = reportdict(dockedliganddict, ["id","source","prep","localfilename"])
print("\n".join(lines))

id               source    prep          localfilename        
EST_redock_1ERE  1ERE.pdb  rdkit_save_H  EST_redock_1ERE_A.sdf
EST_redock_1GWR  1GWR.pdb  rdkit_save_H  EST_redock_1GWR_A.sdf
EST_redock_3UUD  3UUD.pdb  rdkit_save_H  EST_redock_3UUD_A.sdf
EST_redock_6CBZ  6CBZ.pdb  rdkit_save_H  EST_redock_6CBZ_A.sdf
DES_redock_3ERD  3ERD.pdb  rdkit_save_H  DES_redock_3ERD_A.sdf
DES_redock_4ZN7  4ZN7.pdb  rdkit_save_H  DES_redock_4ZN7_A.sdf
27M_redock_4MGC  4MGC.pdb  rdkit_save_H  27M_redock_4MGC_A.sdf
27J_redock_4MG8  4MG8.pdb  rdkit_save_H  27J_redock_4MG8_A.sdf
36J_redock_4TUZ  4TUZ.pdb  rdkit_save_H  36J_redock_4TUZ_A.sdf
2OH_redock_3UU7  3UU7.pdb  rdkit_save_H  2OH_redock_3UU7_A.sdf
27K_redock_4MG9  4MG9.pdb  rdkit_save_H  27K_redock_4MG9_A.sdf
27L_redock_4MGA  4MGA.pdb  rdkit_save_H  27L_redock_4MGA_A.sdf
EST_redock_1G50  1G50.pdb  rdkit_save_H  EST_redock_1G50_A.sdf


## Populate protein dictionary

In [8]:
import json

# Read from text file
with open(prodictjson, "r") as f:
    proteindict = json.load(f)    

lines = reportdict(proteindict, ["id","name","prep","localfilename_fixed"])
print("\n".join(lines))

id    name  prep          localfilename_fixed
1ERE        PDBfixfunc74  1ERE_A_fixed.pdb   
1GWR        PDBfixfunc74  1GWR_A_fixed.pdb   
3UUD        PDBfixfunc74  3UUD_A_fixed.pdb   
6CBZ        PDBfixfunc74  6CBZ_A_fixed.pdb   
3ERD        PDBfixfunc74  3ERD_A_fixed.pdb   
4ZN7        PDBfixfunc74  4ZN7_A_fixed.pdb   
4MGC        PDBfixfunc74  4MGC_A_fixed.pdb   
4MG8        PDBfixfunc74  4MG8_A_fixed.pdb   
4TUZ        PDBfixfunc74  4TUZ_A_fixed.pdb   
3UU7        PDBfixfunc74  3UU7_A_fixed.pdb   
4MG9        PDBfixfunc74  4MG9_A_fixed.pdb   
4MGA        PDBfixfunc74  4MGA_A_fixed.pdb   
1G50        PDBfixfunc74  1G50_A_fixed.pdb   


In [9]:
# Add docked ligand id into protein dictionary. Next iteration this will happen in the 'pre process protein' code.

for (k1, v1), (k2, v2) in zip(proteindict.items(), dockedliganddict.items()):
    #print(f"Key: {k1}, Dict1: {v1}, Dict2: {v2}")
    v1['dockedid'] = v2['id']

lines = reportdict(proteindict, ["id","name","prep","localfilename_fixed","dockedid"])
print("\n".join(lines))

id    name  prep          localfilename_fixed  dockedid       
1ERE        PDBfixfunc74  1ERE_A_fixed.pdb     EST_redock_1ERE
1GWR        PDBfixfunc74  1GWR_A_fixed.pdb     EST_redock_1GWR
3UUD        PDBfixfunc74  3UUD_A_fixed.pdb     EST_redock_3UUD
6CBZ        PDBfixfunc74  6CBZ_A_fixed.pdb     EST_redock_6CBZ
3ERD        PDBfixfunc74  3ERD_A_fixed.pdb     DES_redock_3ERD
4ZN7        PDBfixfunc74  4ZN7_A_fixed.pdb     DES_redock_4ZN7
4MGC        PDBfixfunc74  4MGC_A_fixed.pdb     27M_redock_4MGC
4MG8        PDBfixfunc74  4MG8_A_fixed.pdb     27J_redock_4MG8
4TUZ        PDBfixfunc74  4TUZ_A_fixed.pdb     36J_redock_4TUZ
3UU7        PDBfixfunc74  3UU7_A_fixed.pdb     2OH_redock_3UU7
4MG9        PDBfixfunc74  4MG9_A_fixed.pdb     27K_redock_4MG9
4MGA        PDBfixfunc74  4MGA_A_fixed.pdb     27L_redock_4MGA
1G50        PDBfixfunc74  1G50_A_fixed.pdb     EST_redock_1G50


## Populate ideal ligand dictionary

In [29]:
idealliganddict = {}

In [ ]:
#minimized versions of ideal ligands
idealliganddict["27J_min"] = {'id':"27J_min", 'prep': "obabel-mmff94", 'localfilename': "27J_min.sdf"}
idealliganddict["27K_min"] = {'id':"27K_min", 'prep': "obabel-mmff94", 'localfilename': "27K_min.sdf"}
idealliganddict["27L_min"] = {'id':"27L_min", 'prep': "obabel-mmff94", 'localfilename': "27L_min.sdf"}

idealliganddict["27M_min"] = {'id':"27M_min", 'prep': "obabel-mmff94", 'localfilename': "27M_min.sdf"}
idealliganddict["2OH_min"] = {'id':"2OH_min", 'prep': "obabel-mmff94", 'localfilename': "2OH_min.sdf"}
idealliganddict["36J_min"] = {'id':"36J_min", 'prep': "obabel-mmff94", 'localfilename': "36J_min.sdf"}

idealliganddict["Caffeine_min"] = {'id':"Caffeine_min", 'prep': "obabel-mmff94", 'localfilename': "Caffeine_min.sdf"}
idealliganddict["DES_min"] = {'id':"DES_min", 'prep': "obabel-mmff94", 'localfilename': "DES_min.sdf"}
idealliganddict["EE2_min"] = {'id':"EE2_min", 'prep': "obabel-mmff94", 'localfilename': "EE2_min.sdf"}

idealliganddict["EST_min"] = {'id':"EST_min", 'prep': "obabel-mmff94", 'localfilename': "EST_min.sdf"}
idealliganddict["Melatonin_min"] = {'id':"Melatonin_min", 'prep': "obabel-mmff94", 'localfilename': "Melatonin_min.sdf"}
idealliganddict["Testosterone_min"] = {'id':"Testosterone_min", 'prep': "obabel-mmff94", 'localfilename': "Testosterone_min.sdf"}

In [ ]:
#Ideal ligands
idealliganddict["27J"] = {'id':"27J", 'prep': "obabel-mmff94", 'localfilename': "27J.sdf"}
idealliganddict["27K"] = {'id':"27K", 'prep': "obabel-mmff94", 'localfilename': "27K.sdf"}
idealliganddict["27L"] = {'id':"27L", 'prep': "obabel-mmff94", 'localfilename': "27L.sdf"}

idealliganddict["27M"] = {'id':"27M", 'prep': "obabel-mmff94", 'localfilename': "27M.sdf"}
idealliganddict["2OH"] = {'id':"2OH", 'prep': "obabel-mmff94", 'localfilename': "2OH.sdf"}
idealliganddict["36J"] = {'id':"36J", 'prep': "obabel-mmff94", 'localfilename': "36J.sdf"}

idealliganddict["Caffeine"] = {'id':"Caffeine", 'prep': "obabel-mmff94", 'localfilename': "Caffeine.sdf"}
idealliganddict["DES"] = {'id':"DES", 'prep': "obabel-mmff94", 'localfilename': "DES.sdf"}
idealliganddict["EE2"] = {'id':"EE2", 'prep': "obabel-mmff94", 'localfilename': "EE2.sdf"}

idealliganddict["EST"] = {'id':"EST", 'prep': "obabel-mmff94", 'localfilename': "EST.sdf"}
idealliganddict["Melatonin"] = {'id':"Melatonin", 'prep': "obabel-mmff94", 'localfilename': "Melatonin.sdf"}
idealliganddict["Testosterone"] = {'id':"Testosterone", 'prep': "obabel-mmff94", 'localfilename': "Testosterone.sdf"}

In [30]:
import os
import re

for entry in os.scandir(idealdir):
    if entry.is_file() and entry.name.endswith(".sdf"):
        print ("I see file " + entry.name)
        id = "test" + entry.name
        match = re.search(r'^cas-(.*)\.sdf$', entry.name)
        id = match.group(1)       
        idealliganddict[id] = {'id':id, 'prep': "obabel-mmff94", 'localfilename': entry.name}

I see file cas-50-28-2_min.sdf
I see file cas-34816-55-2_min.sdf
I see file cas-131-57-7_min.sdf
I see file cas-72-43-5_min.sdf
I see file cas-84-16-2_min.sdf
I see file cas-90-43-7_min.sdf
I see file cas-57-63-6_min.sdf
I see file cas-207-13-4_min.sdf
I see file cas-474-86-2_min.sdf
I see file cas-84-17-3_min.sdf
I see file cas-56-53-1_min.sdf
I see file cas-50-27-1_min.sdf
I see file cas-123-07-9_min.sdf
I see file cas-72-33-3_min.sdf
I see file cas-92-69-3_min.sdf
I see file cas-85-68-7_min.sdf
I see file cas-5976-61-4_min.sdf
I see file cas-362-05-0_min.sdf
I see file cas-115-86-6_min.sdf
I see file cas-72-55-9_min.sdf
I see file cas-57-91-0_min.sdf
I see file cas-1570-64-5_min.sdf
I see file cas-72-54-8_min.sdf
I see file cas-53-63-4_min.sdf
I see file cas-140-10-3_min.sdf
I see file cas-59-50-7_min.sdf
I see file cas-53-19-0_min.sdf
I see file cas-99-76-3_min.sdf
I see file cas-53-16-7_min.sdf
I see file cas-115-29-7_min.sdf
I see file cas-84-69-5_min.sdf
I see file cas-120-47-8_

In [32]:
lines = reportdict(idealliganddict, ["id","prep","localfilename"])
print("\n".join(lines))

id              prep           localfilename         
50-28-2_min     obabel-mmff94  cas-50-28-2_min.sdf   
34816-55-2_min  obabel-mmff94  cas-34816-55-2_min.sdf
131-57-7_min    obabel-mmff94  cas-131-57-7_min.sdf  
72-43-5_min     obabel-mmff94  cas-72-43-5_min.sdf   
84-16-2_min     obabel-mmff94  cas-84-16-2_min.sdf   
90-43-7_min     obabel-mmff94  cas-90-43-7_min.sdf   
57-63-6_min     obabel-mmff94  cas-57-63-6_min.sdf   
207-13-4_min    obabel-mmff94  cas-207-13-4_min.sdf  
474-86-2_min    obabel-mmff94  cas-474-86-2_min.sdf  
84-17-3_min     obabel-mmff94  cas-84-17-3_min.sdf   
56-53-1_min     obabel-mmff94  cas-56-53-1_min.sdf   
50-27-1_min     obabel-mmff94  cas-50-27-1_min.sdf   
123-07-9_min    obabel-mmff94  cas-123-07-9_min.sdf  
72-33-3_min     obabel-mmff94  cas-72-33-3_min.sdf   
92-69-3_min     obabel-mmff94  cas-92-69-3_min.sdf   
85-68-7_min     obabel-mmff94  cas-85-68-7_min.sdf   
5976-61-4_min   obabel-mmff94  cas-5976-61-4_min.sdf 
362-05-0_min    obabel-mmff9

## Examine proteins to determine what chains and ligands are present in the proteins. (Optional informational step) ##

In [ ]:
import gemmi

for key, value in proteindict.items():
    
    structure = gemmi.read_structure(prodir + value['localfilename_fixed'])
    #structure = gemmi.read_structure(chemfilesdir + "6O4w_rcbs.pdb")
    ligands = []
    
    for model in structure:
        for chain in model:
            for res in chain:
                if res.het_flag != ' ':  # hetero-residue
                    if res.name not in ("HOH", "WAT", "H2O"):
                        #if res.seqid.num == 604:
                        ligands.append((res.name, chain.name, res.seqid.num))

    print("protein: " + value['localfilename'])
    print(set(ligands))

In [ ]:
from Bio.PDB import PDBParser

for key, value in proteindict.items():
    
  parser = PDBParser(QUIET=True)
  structure = parser.get_structure("prot", prodir + value['localfilename_fixed'])
  #structure = parser.get_structure("prot", prodir + value['localfilename_fixed'])
    
  print("Protein: " + value['localfilename'])
    
  for model in structure:
    print(f"  Model {model.id}:")
    chain_ids = [chain.id for chain in model]
    print("    Chains:", ", ".join(chain_ids))


## Visualize Ligands

In [ ]:
from rdkit import Chem

ligandfile = ligdir + '2R6_ideal_PubChem.sdf'
ligandfileout = ligdir + '2R6_ideal_PubChem_NOH.sdf'

# Load SDF file (remove Hs on read - most efficient)
mol = Chem.MolFromMolFile(ligandfile, removeHs=True)

# Or if already loaded with Hs:
# mol = Chem.MolFromMolFile("ligand.sdf", removeHs=False)
# mol = Chem.RemoveHs(mol)

# Write H-free SDF
writer = Chem.SDWriter(ligandfileout)
writer.write(mol)
writer.close()

print(f"Atoms before: {Chem.MolFromMolFile(ligandfileout, removeHs=False).GetNumAtoms()}")
print(f"Atoms after:  {mol.GetNumAtoms()}")

In [ ]:
import nglview as nv
from rdkit import Chem

# From SDF
view = nv.show_structure_file(ligdir + '2R6_ideal_PubChem.sdf')
view.add_representation('ball+stick')
view.camera = 'orthographic'
view.center()
view

In [ ]:
view = nv.show_structure_file(ligdir + '4o09_final_ligand_2R6_A.pdb')
view.add_representation('ball+stick')
view.camera = 'orthographic'
view.center()
view

In [ ]:
view = nv.show_structure_file(ligdir + '2R6_redock_4o09_final_A_obabel.sdf')
view.add_representation('ball+stick')
view.camera = 'orthographic'
view.center()
view

In [ ]:
view.display(gui=True) 

## Define Gnina and dataframe functions

In [33]:
#Define the functions that call Gnina, parse the results files, and write those results to the dataframe.
from rdkit import Chem
import subprocess
number_of_modes = 1 #Report how many modes from each Gnina run?

def executegnina(proteinid,ligandid,boxid):
    protein = proteindict[proteinid]
    ligand = idealliganddict[ligandid]
    box = dockedliganddict[boxid]
    p = prodir + protein["localfilename_fixed"]
    l = idealdir + ligand["localfilename"]
    b = dockeddir + box["localfilename"]
    return callgnina(p,l,b)


def executegninadecoy(proteinid,boxid):
    protein = proteindict[proteinid]
    box = dockedliganddict[boxid]
    p = prodir + protein["localfilename_fixed"]
    l = decoydir + "decoyligand.sdf"
    b = dockeddir + box["localfilename"]
    return callgnina(p,l,b)

    
def callgnina(p,l,b):
    
    #!~/octoberproject/gnina -r "{p}" -l "{l}" --autobox_ligand "{b}" -o "{docked}" --log "{log}" --exhaustiveness=16 --num_modes=8 --seed 0 --pose_sort_order CNNaffinity --no_gpu  
    #!~/octoberproject/gnina -r "{p}" -l "{l}" --autobox_ligand "{b}" -o "{docked}" --log "{log}" --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu  
    #!"{GNINA_LOC}" -r "{p}" -l "{l}" --autobox_ligand "{b}" -o "{docked}" --log "{log}" --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore "{GNINA_PARAMETER}"

    if not GNINA_PARAMETER:
        gnina_cmd = [GNINA_LOC, "-r", p, "-l", l, "--autobox_ligand", b, "--autobox_add", "4", "-o", docked, "--log", log, 
       "--exhaustiveness=16", "--num_modes=9", "--seed", "0", "--pose_sort_order", "CNNscore"]

    else:
        gnina_cmd = [GNINA_LOC, "-r", p, "-l", l, "--autobox_ligand", b, "--autobox_add", "4", "-o", docked, "--log", log, 
       "--exhaustiveness=16", "--num_modes=9", "--seed", "0", "--pose_sort_order", "CNNscore", 
       GNINA_PARAMETER]

    #return True #To skip the call for Gnina 
    
    print("Call gnina:", " ".join(gnina_cmd))
 
    try:
        result = subprocess.run(gnina_cmd, check=True, capture_output=True, text=True)
        print("stdout:", result.stdout)
    except subprocess.CalledProcessError as e:
        # FAILURE (returncode != 0)
        print("Failure")
        print(f"Return code: {e.returncode}")
        print("stderr:", e.stderr)
        return False
    else:
        #only execute obrms if gnina ran successfully
        #Compute RMSD of two docked ligand files and save results to rmsdlog file.
        !obrms -f "{b}" "{docked}" | tee "{rmsdlog}"
        return True
    
    
def getdockedresultdf(docked):
    rmsddf = pd.read_csv(rmsdlog, sep=" ", header=None)
    rows = []
    for i, mol in enumerate(Chem.SDMolSupplier(docked)):
        if mol is None:
            continue
        rows.append({
            "pose": i,
            "CNNscore": float(mol.GetProp("CNNscore")),
            "CNN_VS": float(mol.GetProp("CNN_VS")),
            "CNNaffinity": float(mol.GetProp("CNNaffinity")),
            "RMSD": float(rmsddf.iloc[i,2]),
        })
    df = pd.DataFrame(rows)
    return(df)

def writeresulttodf(pro,lig,box):
    resultsdf = getdockedresultdf(docked)
    #for index, row in resultsdf.iterrows():
    for index, row in resultsdf.head(number_of_modes).iterrows():
        #write to the resultsdf here.
        temp = [pro, lig, box, "{:.4f}".format(row['CNNscore']), "{:.4f}".format(row['CNN_VS']), "{:.4f}".format(row['RMSD'])]
        outputdf.loc[len(outputdf)] = temp


## Run the docking simulations

In [47]:
# Establish new empty dataframe
import pandas as pd
outputdf = pd.DataFrame(columns = ["protein", "ideal_ligand", "native_ligand", "CNN_pose", "CNN_VS", "RMSD"])

In [48]:
#Test block. Dock just one pair.
pkey = "1ERE"
ikey = "115-29-7_min"
dockedid = "EST_redock_1ERE"
gninasuccess = executegnina(pkey, ikey, dockedid)
if (gninasuccess):
    writeresulttodf(pkey, ikey, dockedid) 

Call gnina: /home/dwaine/octoberproject/gnina -r preprocessed/proteinprep/1ERE_A_fixed.pdb -l preprocessed/minimized_ideal_ligand/cas-115-29-7_min.sdf --autobox_ligand preprocessed/ligandprep/EST_redock_1ERE_A.sdf --autobox_add 4 -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu
stdout:               _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul 29 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /home/dwaine/octoberproject/gnina -r preprocessed/proteinprep/1ERE_A_fixed.pdb -l preprocessed/minimized_ideal_ligand/cas-115-29-7_min.sdf --autobox_ligand preprocessed/ligandprep/EST_redock_1ERE_A.sdf --autobox_add 4 -o results/docked.sdf --log results/gninalog.txt -

[13:08:38] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:08:38] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:08:38] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:08:38] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:08:38] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:08:38] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:08:38] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:08:38] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:08:38] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

In [49]:
# Iterate though protein dictionary and ligand dictionary and perform Gnina docking for every combo.
from itertools import islice

pro = proteindict
#pro = dict(islice(proteindict.items(), 3))
ide = idealliganddict
#ide = dict(islice(idealliganddict.items(), 3))

rundecoys = False

start = time.time()

#iterate through proteins
for pkey, pvalue in pro.items():
    #iterate through ideal ligands
    for ikey, ivalue in ide.items():
        print(f"{pkey} meets {ikey} at {pvalue['dockedid']}")
        executegnina(pkey, ikey, pvalue['dockedid'])
        writeresulttodf(pkey,ikey,pvalue['dockedid'])   

    #iterate through decoy ligands if dataframe exists and is populated
    if (rundecoys):
        for idx, row in decoydf.head(2).iterrows():
            writeonedecoytoFS(row)
            print(f"{pkey} meets {row['ID']} at {pvalue['dockedid']}") 
            executegninadecoy(pkey, pvalue['dockedid'])
            writeresulttodf(pkey, row['ID'], pvalue['dockedid'])  

end = time.time()
elapsed = end - start
print(f"Gnina runs completed in {format_time(elapsed)}")  # 00:01:05

1ERE meets 50-28-2_min at EST_redock_1ERE
Call gnina: /home/dwaine/octoberproject/gnina -r preprocessed/proteinprep/1ERE_A_fixed.pdb -l preprocessed/minimized_ideal_ligand/cas-50-28-2_min.sdf --autobox_ligand preprocessed/ligandprep/EST_redock_1ERE_A.sdf --autobox_add 4 -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu
stdout:               _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul 29 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /home/dwaine/octoberproject/gnina -r preprocessed/proteinprep/1ERE_A_fixed.pdb -l preprocessed/minimized_ideal_ligand/cas-50-28-2_min.sdf --autobox_ligand preprocessed/ligandprep/EST_redock_1ERE_A.sdf --autobox_add 4 -o results

[13:09:30] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:09:30] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:09:30] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:09:30] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:09:30] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:09:30] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:09:30] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:09:30] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:09:30] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

stdout:               _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul 29 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /home/dwaine/octoberproject/gnina -r preprocessed/proteinprep/1ERE_A_fixed.pdb -l preprocessed/minimized_ideal_ligand/cas-34816-55-2_min.sdf --autobox_ligand preprocessed/ligandprep/EST_redock_1ERE_A.sdf --autobox_add 4 -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu
Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
11954041 | pose 0 | initial pose not within box

mode |  affinity  |  intramol  |    CNN     |   

[13:10:12] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:10:12] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:10:12] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:10:12] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:10:12] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:10:12] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:10:12] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:10:12] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:10:12] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

stdout:               _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul 29 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /home/dwaine/octoberproject/gnina -r preprocessed/proteinprep/1ERE_A_fixed.pdb -l preprocessed/minimized_ideal_ligand/cas-131-57-7_min.sdf --autobox_ligand preprocessed/ligandprep/EST_redock_1ERE_A.sdf --autobox_add 4 -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu
Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
4632 | pose 0 | initial pose not within box

mode |  affinity  |  intramol  |    CNN     |   CNN
  

...

[20:01:31] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:01:31] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:01:31] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:01:31] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:01:31] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:01:31] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:01:31] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:01:31] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:01:31] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

stdout:               _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul 29 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /home/dwaine/octoberproject/gnina -r preprocessed/proteinprep/1G50_A_fixed.pdb -l preprocessed/minimized_ideal_ligand/cas-120-47-8_min.sdf --autobox_ligand preprocessed/ligandprep/EST_redock_1G50_A.sdf --autobox_add 4 -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu
Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
8434 | pose 0 | initial pose not within box
8434 | pose 0 | ligand outside box
8434 | pose 0 | liga

[20:02:44] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:02:44] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:02:44] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:02:44] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:02:44] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:02:44] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:02:44] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:02:44] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:02:44] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

stdout:               _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul 29 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /home/dwaine/octoberproject/gnina -r preprocessed/proteinprep/1G50_A_fixed.pdb -l preprocessed/minimized_ideal_ligand/cas-97-54-1_min.sdf --autobox_ligand preprocessed/ligandprep/EST_redock_1G50_A.sdf --autobox_add 4 -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu
Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
853433 | pose 0 | initial pose not within box

mode |  affinity  |  intramol  |    CNN     |   CNN
 

[20:03:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:03:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:03:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:03:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:03:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:03:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:03:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:03:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:03:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

In [50]:
print(f"Gnina runs completed in {format_time(elapsed)}")  # 00:01:05

Gnina runs completed in 06:54:55


In [51]:
# Preview results
print(outputdf)

    protein    ideal_ligand    native_ligand CNN_pose  CNN_VS    RMSD
0      1ERE    115-29-7_min  EST_redock_1ERE   0.7344  5.1837     inf
1      1ERE     50-28-2_min  EST_redock_1ERE   0.9822  8.2022  0.5143
2      1ERE  34816-55-2_min  EST_redock_1ERE   0.9406  7.9502  0.7031
3      1ERE    131-57-7_min  EST_redock_1ERE   0.5081  2.7736     inf
4      1ERE     72-43-5_min  EST_redock_1ERE   0.7792  5.6629     inf
..      ...             ...              ...      ...     ...     ...
425    1G50     53-16-7_min  EST_redock_1G50   0.9914  8.3167  0.5871
426    1G50    115-29-7_min  EST_redock_1G50   0.6604  4.7815     inf
427    1G50     84-69-5_min  EST_redock_1G50   0.6449  3.1626     inf
428    1G50    120-47-8_min  EST_redock_1G50   0.9297  5.3716     inf
429    1G50     97-54-1_min  EST_redock_1G50   0.8767  4.9498     inf

[430 rows x 6 columns]


## Order, rearrance, output as .csv

In [52]:
# Order by LIGAND, then CNN_pose desc. Change order of columns.
outputdfsorted = pd.DataFrame(columns = ["protein", "ideal_ligand", "native_ligand", "CNN_pose", "CNN_VS", "RMSD"])
for group_name, group_df in outputdf.groupby("ideal_ligand"):
    oneliganddf = group_df.sort_values(by="CNN_pose", ascending=False)
    outputdfsorted = pd.concat([outputdfsorted, pd.DataFrame(oneliganddf)], ignore_index=True)
    
outputdfsorted = outputdfsorted[["ideal_ligand", "protein", "native_ligand", "CNN_pose", "CNN_VS", "RMSD"]]

print(outputdfsorted)

     ideal_ligand protein    native_ligand CNN_pose  CNN_VS RMSD
0    115-29-7_min    4MGC  27M_redock_4MGC   0.7838  5.4520  inf
1    115-29-7_min    4MG8  27J_redock_4MG8   0.7749  5.5493  inf
2    115-29-7_min    1ERE  EST_redock_1ERE   0.7344  5.1837  inf
3    115-29-7_min    1ERE  EST_redock_1ERE   0.7344  5.1837  inf
4    115-29-7_min    4MGA  27L_redock_4MGA   0.7008  5.0329  inf
..            ...     ...              ...      ...     ...  ...
425   99-76-3_min    4MGC  27M_redock_4MGC   0.8841  4.5378  inf
426   99-76-3_min    4MG8  27J_redock_4MG8   0.8789  4.1365  inf
427   99-76-3_min    3UU7  2OH_redock_3UU7   0.8780  4.1369  inf
428   99-76-3_min    4MG9  27K_redock_4MG9   0.8738  4.5872  inf
429   99-76-3_min    4TUZ  36J_redock_4TUZ   0.8635  4.2545  inf

[430 rows x 6 columns]


In [ ]:
# Order by PROTEIN, then CNN_pose desc. Change order of columns.
outputdfsorted = pd.DataFrame(columns = ["protein", "ideal_ligand", "native_ligand", "CNN_pose", "CNN_VS", "RMSD"])
for group_name, group_df in outputdf.groupby("protein"):
    oneliganddf = group_df.sort_values(by="CNN_pose", ascending=False)
    outputdfsorted = pd.concat([outputdfsorted, pd.DataFrame(oneliganddf)], ignore_index=True)
    
outputdfsorted = outputdfsorted[["protein", "ideal_ligand", "native_ligand", "CNN_pose", "CNN_VS", "RMSD"]]

print(outputdfsorted)

In [53]:
# Write to .csv
import csv
import time
epoch = int(time.time())
outfile = resultsdir + "results" + str(epoch) + ".csv"
outputdfsorted.to_csv(outfile, mode='w', index=False, header=True)